# Neural Networks – Simple, Intuitive Introduction

**Goal of this notebook**

By the end, I should clearly understand:

- Why do we need neural networks?
- What is a neuron?
- What are weights and bias?
- What is a layer (input, hidden, output)?
- What is an activation function? Why do we need it?
- Types of activation functions (ReLU, Sigmoid, Tanh, Softmax – basic idea)
- What is a forward pass?
- What is a loss function (very simple idea)?
- How training roughly works (high level: backprop + gradient descent)
- Important small concepts: parameters, hyperparameters, overfitting (basic idea)

I should be able to explain these in simple language to another person.


In [1]:
import numpy as np
import matplotlib.pyplot as plt


## 1. Why do we need neural networks?

Traditional models like **linear regression** or simple decision rules work well when:

- Relationships are simple (almost straight line)
- Data is not too complex

But real-world problems are messy:

- Image recognition: “Is there a cat in this picture?”
- Language: “Generate a reply to this message.”
- Speech: “Convert voice to text.”
- Recommendations: “What should this user watch next?”

These problems are:

- **Non-linear** (not straight line)
- **High-dimensional** (many features)
- Often **hard to write rules for** by hand

Neural networks are good because they can:

- Learn **complex patterns** from data
- Approximate many kinds of functions
- Stack many simple units (neurons) to learn powerful behaviors

A simple way to think:

> A neural network is just a big function that takes input, does many small steps, and gives an output.  
> We *learn* the best version of that function from data.


## 2. What is a neuron?

A **neuron** is the basic building block of a neural network.

It does 3 simple things:

1. Takes some inputs (numbers)
2. Combines them using weights and a bias
3. Passes the result through an activation function

Mathematically for one neuron:

- Input: **x** = `[x₁, x₂, ..., xₙ]`
- Weights: **w** = `[w₁, w₂, ..., wₙ]`
- Bias: **b** = a single number
- Linear part:  
  **z = w·x + b = w₁x₁ + w₂x₂ + … + wₙxₙ + b**
- Activation part:  
  **a = f(z)** where f is the activation function (ReLU, sigmoid, etc.)

So:

> **Neuron output = activation(linear(inputs))**


In [3]:
def neuron(x, w, b):
    """
    Simple neuron: linear combination w·x + b (no activation yet)
    x: 1D array of inputs
    w: 1D array of weights
    b: scalar bias
    """
    x = np.array(x)
    w = np.array(w)
    z = np.dot(x, w) + b
    return z

x = [1.0, 2.0]   
w = [0.5, -0.2]  
b = 0.1          

z = neuron(x, w, b)
print("z (linear output) =", z)


z (linear output) = 0.19999999999999998


## 3. What are weights and bias (intuitively)?

- **Weights** tell the neuron how important each input is.
  - Big positive weight → increases output a lot when input is big
  - Big negative weight → decreases output when input is big
- **Bias** is like a base value or starting point.
  - It shifts the output up or down.

Think of:

- Weights = "knobs" controlling strength of each input
- Bias = "default score" before any input

During **training**, we adjust weights and bias to reduce error.
These are called **parameters** of the model.


## 4. What is a layer?

A **layer** is just a group of neurons working at the same time.

Types of layers:

1. **Input layer**
   - Not really doing computation
   - Just holds the input features (e.g., height, weight, age)

2. **Hidden layers**
   - Main workers
   - Each hidden layer takes numbers from previous layer, does:
     - `z = W·x + b`
     - `a = activation(z)`

3. **Output layer**
   - Final result
   - Depends on task:
     - Regression (predict a number) → often linear or no activation
     - Binary classification → sigmoid
     - Multi-class classification → softmax

Visually (small example):

Input (3 numbers) → Hidden (4 neurons) → Output (1 neuron)

Each arrow has a weight. Each neuron has its own bias.


In [4]:
def dense_layer(X, W, b, activation=None):
    """
    X: input (batch_size, input_dim)
    W: weights (input_dim, output_dim)
    b: bias (1, output_dim)
    activation: function to apply element-wise
    """
    Z = X.dot(W) + b
    if activation:
        return activation(Z)
    return Z


X = np.array([[1.0, 2.0, 3.0]])


W1 = np.random.randn(3, 4)
b1 = np.zeros((1, 4))

print("Layer output shape:", dense_layer(X, W1, b1).shape)


Layer output shape: (1, 4)


## 5. Why do we need activation functions?

If we only do:

> Linear → Linear → Linear

Example:

- Layer 1: `z₁ = W₁x + b₁`
- Layer 2: `z₂ = W₂z₁ + b₂`

Mathematically, the combination is still **just a linear function of x**.

So:

- 1 linear layer or 10 linear layers = same power  
  → This is not enough to learn complex patterns.

**Activation functions** add non-linearity.

They allow the network to:

- Bend, curve, and shape the function
- Learn complex decision boundaries (not just straight lines)

So the pattern becomes:

> Linear → Activation → Linear → Activation → ... → Output


## 6. Types of activation functions (and when they are used)

### 6.1 ReLU (Rectified Linear Unit)

**Formula:**
> ReLU(z) = max(0, z)

**Intuition:**

- If z is positive → keep it
- If z is negative or 0 → output 0

**Why used:**

- Very simple and fast
- Works well in deep networks
- Reduces vanishing gradient problems compared to sigmoid/tanh

Used mainly in **hidden layers**.


### 6.2 Sigmoid

**Formula:**
> σ(z) = 1 / (1 + e^(−z))

**Output range:**

- Between 0 and 1

**Why used:**

- Good for **probabilities**  
- Often used in the **output layer for binary classification**  
  (e.g., "spam" vs "not spam").

Problem: For very big |z|, gradients become very small → training can be slow.


### 6.3 Tanh

**Formula:**
> tanh(z) = (e^z − e^(−z)) / (e^z + e^(−z))

**Output range:**

- From −1 to 1

**Intuition:**

- Like sigmoid but centered at 0
- Sometimes better than sigmoid in hidden layers


### 6.4 Softmax (just idea for multi-class)

Softmax is used in the **output layer when we have multiple classes** (e.g., cat, dog, horse).

- Takes a vector of scores
- Converts them to probabilities that sum to 1

We won't code it deeply here, but good to know its purpose.


In [5]:
def relu(z):
    return np.maximum(0, z)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def tanh_fn(z):
    return np.tanh(z)

zs = np.array([-2, -1, 0, 1, 2], dtype=float)

print("z:      ", zs)
print("ReLU:   ", relu(zs))
print("Sigmoid:", sigmoid(zs))
print("Tanh:   ", tanh_fn(zs))


z:       [-2. -1.  0.  1.  2.]
ReLU:    [0. 0. 0. 1. 2.]
Sigmoid: [0.11920292 0.26894142 0.5        0.73105858 0.88079708]
Tanh:    [-0.96402758 -0.76159416  0.          0.76159416  0.96402758]


## 7. What is a forward pass?

A **forward pass** is one complete trip from:

> Input → all layers → Output

At each layer, we do:

1. Linear step: `Z = X·W + b`
2. Activation step: `A = activation(Z)`
3. Pass A to the next layer

Finally, we get output `Y_pred`.

This is what happens when the network **makes a prediction**.
No learning happens here – just computation with current weights.


In [7]:

X = np.array([[1.0, 2.0, 3.0]])


W1 = np.random.randn(3, 4)
b1 = np.zeros((1, 4))

Z1 = X.dot(W1) + b1
A1 = relu(Z1)


W2 = np.random.randn(4, 1)
b2 = np.zeros((1, 1))

Z2 = A1.dot(W2) + b2
Y_pred = sigmoid(Z2)

print("Z1:", Z1)
print("A1 (after ReLU):", A1)
print("Z2:", Z2)
print("Y_pred (final prediction):", Y_pred)


Z1: [[-1.83339726  3.2383619  -7.57287544  4.49843788]]
A1 (after ReLU): [[0.         3.2383619  0.         4.49843788]]
Z2: [[-5.76831658]]
Y_pred (final prediction): [[0.00311528]]


## 8. What is a loss function?

A **loss function** tells us:

> How bad are our predictions compared to the true answers?

We want:

- Low loss = good
- High loss = bad

Examples:

- **Mean Squared Error (MSE)** – for regression  
  `Loss = average( (y_pred − y_true)² )`
- **Binary Cross Entropy** – for binary classification  
  (used with sigmoid)

We don't need deep math now.  
Just remember:

> Loss = "score of how wrong we are"  
> Training = "change weights to make this score small"


In [8]:
y_true = np.array([[1.0]])     
y_pred = np.array([[0.7]])     


loss = np.mean((y_pred - y_true)**2)
print("Loss (MSE):", loss)


Loss (MSE): 0.09000000000000002


## 9. How does training work? (high-level)

Training a neural network is mainly:

1. Take a batch of data (X, y)
2. **Forward pass**: compute predictions y_pred
3. Compute **loss**: how wrong are we?
4. **Backpropagation**:
   - Compute gradients: how should we change each weight to reduce loss?
5. **Gradient Descent** step:
   - Update weights:
     - `w_new = w_old − learning_rate * gradient`

Repeat many times.

You don't need full formulas now. Focus on intuition:

- Backprop = a smart way to apply chain rule of calculus through all layers
- Gradient = direction of steepest increase, so we move in opposite direction

Important hyperparameter:

- **Learning rate**: how big a step we take when updating weights
  - Too big → unstable, jumps around
  - Too small → very slow learning


## 10. Parameters vs Hyperparameters

**Parameters** (learned from data):

- Weights (W)
- Biases (b)

The model changes them during training.

**Hyperparameters** (you choose them, not learned directly):

- Learning rate
- Number of layers
- Number of neurons per layer
- Batch size
- Number of training epochs
- Choice of activation function
- Choice of optimizer (SGD, Adam, etc.)

Tuning hyperparameters is a big part of making neural networks work well.


## 11. Overfitting vs Underfitting (simple idea)

- **Underfitting**:
  - Model is too simple
  - Cannot learn the pattern
  - Bad performance on both training and test data

- **Overfitting**:
  - Model memorizes the training data
  - Very good on training data
  - Bad on new (test) data

Neural networks are powerful and can easily **overfit** if:

- Too many parameters
- Not enough data
- No regularization

Common ways to reduce overfitting (just names for now):

- Use more data
- Regularization (L2, dropout)
- Early stopping
- Simpler model


## 12. Final summary checklist

I should now understand:

- [ ] Why we need neural networks (for complex, non-linear problems)
- [ ] What a neuron is:
  - Inputs → weights + bias → linear z → activation → output
- [ ] What weights and bias mean
- [ ] What a layer is (input, hidden, output)
- [ ] Why activation functions are needed (non-linearity)
- [ ] Types of activation functions:
  - ReLU (hidden layers)
  - Sigmoid (probabilities / binary output)
  - Tanh (centered, sometimes hidden)
  - Softmax (multi-class output)
- [ ] What a forward pass is
- [ ] What a loss function is (measures error)
- [ ] Very high-level idea of:
  - Backpropagation (compute gradients)
  - Gradient descent (update weights)
- [ ] Difference between parameters and hyperparameters
- [ ] Basic idea of overfitting vs underfitting

If any box is not clear, I should scroll back to that section and re-read + run the small code examples.
